In [ ]:
%load_ext autoreload
%autoreload 2

from src import GraspAnalysisResults, GraspRegion, InitializationAngle
from src.utils import GraspAnalysisUtils
from pathlib import Path
import numpy as np


In [ ]:
# Find folder containing CSV files for analysis
data_folder = GraspAnalysisUtils.select_folder()


In [ ]:
# Check if there as csv files in the chosen directory
dir_path = Path(data_folder)
if any(file.suffix.lower() == ".csv" for file in dir_path.iterdir() if file.is_file()):
    files = [str(file) for file in dir_path.glob("*.csv")]
    print("CSV files located")
else:
    raise RuntimeError("No CSV files found in chosen folder")

In [ ]:
# Plot Dialog Selection
chosen_plots = GraspAnalysisUtils.select_desired_plots()
print(f"User wants to create: {chosen_plots}")

In [ ]:
# Assignment of variables
SAMPLE_SIZE = 200
OFFSET = 2000
MIN_FORCE_THRESHOLD = 10

In [ ]:
# Performing Analysis

for file_path in files:

    result = GraspAnalysisResults
    full_path = Path(file_path)

    result.filename = full_path.name

    force_data = GraspAnalysisUtils.load_and_preprocess_data(file_path)

    grasps = GraspAnalysisUtils.detect_grasp_regions(force_data, SAMPLE_SIZE, MIN_FORCE_THRESHOLD)

    result.number_of_grasps = len(grasps)

    result.rolling_avg = []
    result.rolling_std = []
    result.rolling_median = []

    if result.number_of_grasps == 0:
        Warning(f"No grasps detected in file: {result.filename}")
        continue

    avg_forces = []
    for grasp in grasps:
        grasp = GraspAnalysisUtils.calculate_grasp_force(force_data[grasp.start_idx:grasp.end_idx], grasp, OFFSET)

        avg_forces.append(grasp.avg_force)
        result.rolling_avg.append(np.mean(avg_forces))
        result.rolling_std.append(np.std(avg_forces))
        result.rolling_median.append(np.median(avg_forces))

    # Calculate Statistics 
    result = GraspAnalysisUtils.calculate_grasp_statistics(result, grasps)

    result = GraspAnalysisUtils.calculate_required_samples(result)

    GraspAnalysisUtils.report_results(result, grasps)

    GraspAnalysisUtils.create_grasp_plots(chosen_plots, force_data, result, grasps)